# OMQS WebSocket API Notebook Client

This notebook connects to the OMQS WebSocket API, authenticates with a `.env` key, subscribes to example pairs, and prints live data every 5 seconds.

Required `.env` variables:

```text
OMQS_WS_API_KEY=your_websocket_api_key_here
OMQS_WS_URL=wss://om-qs.com/ws/api/v1/models/
```

The notebook stores live data in normal Python dictionaries, not in pandas DataFrames.


In [ ]:
# Cell 1: connect, authenticate, subscribe, receive data in background

import os
import json
import threading
from datetime import datetime
from collections import defaultdict, deque

import websocket
from dotenv import load_dotenv

load_dotenv(override=True)

OMQS_WS_API_KEY = os.getenv('OMQS_WS_API_KEY')
OMQS_WS_URL = os.getenv('OMQS_WS_URL', 'wss://om-qs.com/ws/api/v1/models/')

if not OMQS_WS_API_KEY:
    raise ValueError('OMQS_WS_API_KEY not found. Please check your .env file.')

OMQS_PAIRS = [
    {'model': 'crypto', 'ticker': 'BTCUSDT', 'timeframe': '1m'},
    {'model': 'crypto', 'ticker': 'ETHUSDT', 'timeframe': '5m'},
    {'model': 'forex', 'ticker': 'EURUSD', 'timeframe': '5m'},
]

omqs_lock = threading.Lock()
omqs_latest = {}
omqs_history = defaultdict(lambda: deque(maxlen=500))
omqs_status = {
    'connected': False,
    'authenticated': False,
    'subscribed': False,
    'last_message_at': None,
    'last_error': None,
    'connection_id': None,
}

def now_utc_string():
    return datetime.utcnow().strftime('%Y-%m-%d %H:%M:%S UTC')

def normalize_number(value):
    if value is None:
        return None
    try:
        return float(value)
    except Exception:
        return value

def signal_label(value):
    if value == 1:
        return 'blue / long-bias regime'
    if value == -1:
        return 'red / short-bias regime'
    return 'signal unavailable'

def normalize_candle_message(message):
    stops = message.get('stops') or {}
    timestamp = message.get('timestamp')
    candle_datetime = None
    if timestamp is not None:
        candle_datetime = datetime.fromtimestamp(timestamp).strftime('%Y-%m-%d %H:%M:%S')
    return {
        'received_at': now_utc_string(),
        'symbol': message.get('symbol'),
        'timeframe': message.get('timeframe'),
        'timestamp': timestamp,
        'datetime': candle_datetime,
        'open': normalize_number(message.get('open')),
        'high': normalize_number(message.get('high')),
        'low': normalize_number(message.get('low')),
        'close': normalize_number(message.get('close')),
        'volume': normalize_number(message.get('volume')),
        'closed': message.get('closed'),
        'omqs': message.get('omqs'),
        'signal_label': signal_label(message.get('omqs')),
        'stop': normalize_number(message.get('stop')),
        'stop_fast': normalize_number(stops.get('fast')),
        'stop_balanced': normalize_number(stops.get('balanced')),
        'stop_slow': normalize_number(stops.get('slow')),
    }

def store_candle(candle):
    key = (candle['symbol'], candle['timeframe'])
    with omqs_lock:
        omqs_latest[key] = candle
        omqs_history[key].append(candle)
        omqs_status['last_message_at'] = now_utc_string()

def get_status():
    with omqs_lock:
        return dict(omqs_status)

def get_latest_data():
    with omqs_lock:
        return list(omqs_latest.values())

def get_history_data(symbol=None, timeframe=None):
    rows = []
    with omqs_lock:
        for key, values in omqs_history.items():
            key_symbol, key_timeframe = key
            if symbol is not None and key_symbol != symbol:
                continue
            if timeframe is not None and key_timeframe != timeframe:
                continue
            rows.extend(list(values))
    return rows

def omqs_websocket_loop():
    ws = None
    try:
        ws = websocket.create_connection(OMQS_WS_URL, timeout=15)
        with omqs_lock:
            omqs_status['connected'] = True
            omqs_status['last_error'] = None

        while True:
            raw_message = ws.recv()
            message = json.loads(raw_message)
            message_type = message.get('type')
            with omqs_lock:
                omqs_status['last_message_at'] = now_utc_string()

            if message_type == 'hello':
                ws.send(json.dumps({'type': 'auth', 'api_key': OMQS_WS_API_KEY}))
            elif message_type == 'ready':
                with omqs_lock:
                    omqs_status['authenticated'] = True
                    omqs_status['connection_id'] = message.get('connection_id')
                ws.send(json.dumps({'type': 'subscribe', 'request_id': 'notebook-main-subscription', 'pairs': OMQS_PAIRS}))
            elif message_type == 'subscribed':
                with omqs_lock:
                    omqs_status['subscribed'] = True
            elif message_type == 'ping':
                ws.send(json.dumps({'type': 'pong'}))
            elif message_type == 'snapshot':
                symbol = message.get('symbol')
                timeframe = message.get('timeframe')
                for item in message.get('ohlc', []):
                    item['symbol'] = symbol
                    item['timeframe'] = timeframe
                    store_candle(normalize_candle_message(item))
            elif message_type == 'candle':
                store_candle(normalize_candle_message(message))
            elif message_type == 'error':
                with omqs_lock:
                    omqs_status['last_error'] = message
                print('OMQS error:', message)
    except Exception as e:
        with omqs_lock:
            omqs_status['connected'] = False
            omqs_status['last_error'] = str(e)
        print('WebSocket stopped with error:', e)
    finally:
        if ws is not None:
            try:
                ws.close()
            except Exception:
                pass
        with omqs_lock:
            omqs_status['connected'] = False

omqs_thread = threading.Thread(target=omqs_websocket_loop, daemon=True)
omqs_thread.start()

print('OMQS WebSocket client started in background.')
print('Use get_status(), get_latest_data(), or get_history_data().')


In [ ]:
# Cell 2: print latest OMQS results every 5 seconds, no DataFrame

import time
from IPython.display import clear_output

while True:
    clear_output(wait=True)
    status = get_status()
    latest_data = get_latest_data()

    print('OMQS WebSocket status')
    print('---------------------')
    print(f"Connected:     {status.get('connected')}")
    print(f"Authenticated: {status.get('authenticated')}")
    print(f"Subscribed:    {status.get('subscribed')}")
    print(f"Connection ID: {status.get('connection_id')}")
    print(f"Last message:  {status.get('last_message_at')}")
    print(f"Last error:    {status.get('last_error')}")
    print()

    if not latest_data:
        print('Waiting for data...')
    else:
        print('Latest OMQS data')
        print('----------------')
        for item in sorted(latest_data, key=lambda x: (x.get('symbol') or '', x.get('timeframe') or '')):
            print(
                f"{item.get('symbol')} | TF {item.get('timeframe')} | "
                f"Close: {item.get('close')} | "
                f"OMQS: {item.get('omqs')} ({item.get('signal_label')}) | "
                f"Closed: {item.get('closed')} | "
                f"Stop: {item.get('stop')}"
            )

    time.sleep(5)
